In [1]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
combined_path = os.path.join(
    workDir,
    r"_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif"
)

print("="*70)
print("ANALYSE DES KOMBINIERTEN WOODY-FIRE DATENSATZES")
print("="*70)

# 1. GRUNDLEGENDE RASTER-INFORMATIONEN
print("\n1. RASTER-EIGENSCHAFTEN")
print("-" * 70)

with rasterio.open(combined_path) as src:
    print(f"Dateipfad: {combined_path}")
    print(f"Dateigröße: {os.path.getsize(combined_path) / (1024**3):.2f} GB")
    print(f"Anzahl Bänder: {src.count}")
    print(f"Dimensionen: {src.height} × {src.width} Pixel")
    print(f"Datentyp: {src.dtypes[0]}")
    print(f"NoData-Wert: {src.nodata}")
    print(f"CRS: {src.crs}")
    print(f"Auflösung: {src.res[0]} × {src.res[1]} m")
    print(f"Bounding Box: {src.bounds}")
    print(f"Kompression: {src.profile.get('compress', 'keine')}")
    
    # Speichere Metadaten für spätere Nutzung
    band_descriptions = src.descriptions
    height, width = src.height, src.width
    total_pixels = height * width

# 2. BANDSTRUKTUR ANALYSIEREN
print("\n2. BANDSTRUKTUR")
print("-" * 70)

# Extrahiere Woody Cover Jahre
woody_years = []
mba_info = []

for i, desc in enumerate(band_descriptions, 1):
    if desc and "Woody_Cover_" in desc:
        year = int(desc.split("_")[-1])
        woody_years.append((i, year))
    elif desc and "MBA_" in desc:
        parts = desc.split("_")
        year = int(parts[1])
        metric = "_".join(parts[2:])
        mba_info.append((i, year, metric))

print(f"Woody Cover Bänder: {len(woody_years)} ({woody_years[0][1]}-{woody_years[-1][1]})")
print(f"MBA Metriken Bänder: {len(mba_info)} (26 Jahre × 12 Metriken)")

# 3. STATISTIKEN PRO BAND BERECHNEN
print("\n3. BAND-STATISTIKEN BERECHNEN")
print("-" * 70)
print("Berechne Statistiken für alle Bänder (kann einige Minuten dauern)...")

stats_list = []

with rasterio.open(combined_path) as src:
    # Woody Cover Statistiken
    print("\nBerechne Woody Cover Statistiken...")
    for band_idx, year in tqdm(woody_years[:5], desc="Woody (sample)"):  # Nur erste 5 als Sample
        band = src.read(band_idx, masked=True)
        stats_list.append({
            'Band': band_idx,
            'Type': 'Woody_Cover',
            'Year': year,
            'Metric': 'cover',
            'Min': float(band.min()),
            'Max': float(band.max()),
            'Mean': float(band.mean()),
            'Std': float(band.std()),
            'Valid_Pixels': int(band.count()),
            'Valid_Percent': float(band.count() / total_pixels * 100)
        })
    
    # MBA Burned Statistiken (nur burned-Bänder)
    print("\nBerechne MBA Burned Statistiken...")
    burned_bands = [info for info in mba_info if info[2] == 'burned']
    for band_idx, year, metric in tqdm(burned_bands, desc="MBA Burned"):
        band = src.read(band_idx, masked=True)
        burned_pixels = int((band == 1).sum())
        stats_list.append({
            'Band': band_idx,
            'Type': 'MBA',
            'Year': year,
            'Metric': metric,
            'Min': 0,
            'Max': 1,
            'Mean': float(band.mean()),
            'Std': float(band.std()),
            'Valid_Pixels': int(band.count()),
            'Burned_Pixels': burned_pixels,
            'Burned_Percent': float(burned_pixels / total_pixels * 100)
        })

df_stats = pd.DataFrame(stats_list)

# 4. WOODY COVER ZEITREIHE
print("\n4. WOODY COVER ENTWICKLUNG")
print("-" * 70)

woody_df = df_stats[df_stats['Type'] == 'Woody_Cover'].copy()
if not woody_df.empty:
    print(f"Durchschnittliche Woody Cover über alle Jahre:")
    print(f"  Min: {woody_df['Mean'].min():.2f}%")
    print(f"  Max: {woody_df['Mean'].max():.2f}%")
    print(f"  Mittelwert: {woody_df['Mean'].mean():.2f}%")
    print(f"  Trend: {woody_df['Mean'].iloc[-1] - woody_df['Mean'].iloc[0]:+.2f}% (Gesamt)")

# 5. FIRE ACTIVITY ANALYSE
print("\n5. FEUER-AKTIVITÄT ÜBER ZEIT")
print("-" * 70)

fire_df = df_stats[df_stats['Metric'] == 'burned'].copy()
if not fire_df.empty:
    fire_df = fire_df.sort_values('Year')
    
    print("Jährliche verbrannte Fläche:")
    for _, row in fire_df.iterrows():
        burned_km2 = row['Burned_Pixels'] * (500**2) / 1e6  # 500m Auflösung
        print(f"  {int(row['Year'])}: {burned_km2:>8,.0f} km² "
              f"({row['Burned_Percent']:.3f}% des Studiengebiets)")
    
    total_burned = fire_df['Burned_Pixels'].sum()
    total_burned_km2 = total_burned * (500**2) / 1e6
    avg_burned_km2 = fire_df['Burned_Pixels'].mean() * (500**2) / 1e6
    
    print(f"\nZusammenfassung:")
    print(f"  Gesamt verbrannte Fläche (2000-2025): {total_burned_km2:,.0f} km²")
    print(f"  Durchschnitt pro Jahr: {avg_burned_km2:,.0f} km²")
    print(f"  Maximum (Jahr {fire_df.loc[fire_df['Burned_Pixels'].idxmax(), 'Year']:.0f}): "
          f"{fire_df['Burned_Pixels'].max() * (500**2) / 1e6:,.0f} km²")
    print(f"  Minimum (Jahr {fire_df.loc[fire_df['Burned_Pixels'].idxmin(), 'Year']:.0f}): "
          f"{fire_df['Burned_Pixels'].min() * (500**2) / 1e6:,.0f} km²")

# 6. SPEICHERE STATISTIKEN
print("\n6. EXPORTIERE STATISTIKEN")
print("-" * 70)

output_dir = os.path.join(workDir, "_Analysis", "03_Woody_Fire_combined", "Analysis")
os.makedirs(output_dir, exist_ok=True)

stats_csv = os.path.join(output_dir, "band_statistics.csv")
df_stats.to_csv(stats_csv, index=False)
print(f"✓ Statistiken gespeichert: {stats_csv}")

fire_csv = os.path.join(output_dir, "fire_activity_yearly.csv")
fire_df.to_csv(fire_csv, index=False)
print(f"✓ Feuer-Aktivität gespeichert: {fire_csv}")

# 7. VISUALISIERUNGEN
print("\n7. ERSTELLE VISUALISIERUNGEN")
print("-" * 70)

# Plot 1: Feuer-Aktivität über Zeit
if not fire_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    fire_df['Burned_km2'] = fire_df['Burned_Pixels'] * (500**2) / 1e6
    ax.bar(fire_df['Year'], fire_df['Burned_km2'], color='orangered', alpha=0.7)
    ax.set_xlabel('Jahr', fontsize=12)
    ax.set_ylabel('Verbrannte Fläche (km²)', fontsize=12)
    ax.set_title('Jährliche Feuer-Aktivität (2000-2025)', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plot1 = os.path.join(output_dir, "fire_activity_timeline.png")
    plt.savefig(plot1, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Plot erstellt: {plot1}")

# Plot 2: Woody Cover Entwicklung (wenn Sample vorhanden)
if not woody_df.empty and len(woody_df) > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(woody_df['Year'], woody_df['Mean'], marker='o', linewidth=2, color='forestgreen')
    ax.fill_between(woody_df['Year'], 
                     woody_df['Mean'] - woody_df['Std'], 
                     woody_df['Mean'] + woody_df['Std'], 
                     alpha=0.3, color='forestgreen')
    ax.set_xlabel('Jahr', fontsize=12)
    ax.set_ylabel('Durchschnittliche Woody Cover (%)', fontsize=12)
    ax.set_title('Woody Cover Entwicklung (Sample)', fontsize=14, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plot2 = os.path.join(output_dir, "woody_cover_trend.png")
    plt.savefig(plot2, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Plot erstellt: {plot2}")

print("\n" + "="*70)
print("✓✓✓ ANALYSE ABGESCHLOSSEN! ✓✓✓")
print("="*70)
print(f"Ergebnisse gespeichert in: {output_dir}")
print("="*70)

ANALYSE DES KOMBINIERTEN WOODY-FIRE DATENSATZES

1. RASTER-EIGENSCHAFTEN
----------------------------------------------------------------------
Dateipfad: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif
Dateigröße: 1.44 GB
Anzahl Bänder: 352
Dimensionen: 9660 × 10483 Pixel
Datentyp: uint8
NoData-Wert: 255.0
CRS: EPSG:3035
Auflösung: 500.0 × 500.0 m
Bounding Box: BoundingBox(left=2477500.0, bottom=1289000.0, right=7719000.0, top=6119000.0)
Kompression: lzw

2. BANDSTRUKTUR
----------------------------------------------------------------------
Woody Cover Bänder: 40 (1985-2024)
MBA Metriken Bänder: 312 (26 Jahre × 12 Metriken)

3. BAND-STATISTIKEN BERECHNEN
----------------------------------------------------------------------
Berechne Statistiken für alle Bänder (kann einige Minuten dauern)...

Berechne Woody Cover Statistiken...


Woody (sample): 100%|██████████| 5/5 [02:18<00:00, 27.74s/it]



Berechne MBA Burned Statistiken...


MBA Burned: 100%|██████████| 26/26 [12:05<00:00, 27.91s/it]



4. WOODY COVER ENTWICKLUNG
----------------------------------------------------------------------
Durchschnittliche Woody Cover über alle Jahre:
  Min: 10.79%
  Max: 12.22%
  Mittelwert: 11.56%
  Trend: +1.43% (Gesamt)

5. FEUER-AKTIVITÄT ÜBER ZEIT
----------------------------------------------------------------------
Jährliche verbrannte Fläche:
  2000:      535 km² (0.002% des Studiengebiets)
  2001:  135,321 km² (0.535% des Studiengebiets)
  2002:  122,597 km² (0.484% des Studiengebiets)
  2003:   59,490 km² (0.235% des Studiengebiets)
  2004:   78,826 km² (0.311% des Studiengebiets)
  2005:  124,610 km² (0.492% des Studiengebiets)
  2006:  128,340 km² (0.507% des Studiengebiets)
  2007:  109,226 km² (0.431% des Studiengebiets)
  2008:  143,319 km² (0.566% des Studiengebiets)
  2009:  116,057 km² (0.458% des Studiengebiets)
  2010:  100,118 km² (0.395% des Studiengebiets)
  2011:   66,279 km² (0.262% des Studiengebiets)
  2012:   54,170 km² (0.214% des Studiengebiets)
  2013:   34,

In [2]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
combined_path = os.path.join(
    workDir,
    r"_Runs\03_Woody_Fire_combined\woody_burned_MBA_full_combined.tif"
)

print("="*70)
print("FEUER-WOODY COVER BEZIEHUNGS-ANALYSE")
print("="*70)

# Analysiere für ein bestimmtes Jahr (z.B. 2020)
analysis_year = 2020

print(f"\nAnalysiere Brand-Jahr: {analysis_year}")

with rasterio.open(combined_path) as src:
    # Finde relevante Bänder
    woody_year_idx = 2020 - 1985 + 1  # Woody Cover 2020 (Band 36)
    mba_base_idx = 40 + ((2020 - 2000) * 12)  # MBA 2020 Start
    
    # Lade Daten
    woody_2020 = src.read(woody_year_idx, masked=True)
    burned_2020 = src.read(mba_base_idx + 1, masked=True)  # burned
    count_2020 = src.read(mba_base_idx + 2, masked=True)   # count
    
    print(f"\n1. DATENÜBERSICHT {analysis_year}")
    print(f"   Woody Cover: Band {woody_year_idx}")
    print(f"   Burned: Band {mba_base_idx + 1}")
    print(f"   Count: Band {mba_base_idx + 2}")

# 2. WOODY COVER IN VERBRANNTEN VS. NICHT-VERBRANNTEN GEBIETEN
print(f"\n2. WOODY COVER VERGLEICH")
print("-" * 70)

burned_mask = burned_2020 == 1
not_burned_mask = burned_2020 == 0

woody_burned = woody_2020[burned_mask]
woody_not_burned = woody_2020[not_burned_mask]

print(f"Verbrannte Pixel: {burned_mask.sum():,}")
print(f"Nicht-verbrannte Pixel: {not_burned_mask.sum():,}")

if len(woody_burned) > 0:
    print(f"\nWoody Cover in verbrannten Gebieten:")
    print(f"  Mittelwert: {woody_burned.mean():.2f}%")
    print(f"  Median: {np.median(woody_burned):.2f}%")
    print(f"  Min-Max: {woody_burned.min():.2f}% - {woody_burned.max():.2f}%")

if len(woody_not_burned) > 0:
    print(f"\nWoody Cover in nicht-verbrannten Gebieten:")
    print(f"  Mittelwert: {woody_not_burned.mean():.2f}%")
    print(f"  Median: {np.median(woody_not_burned):.2f}%")
    print(f"  Min-Max: {woody_not_burned.min():.2f}% - {woody_not_burned.max():.2f}%")

# 3. VERTEILUNG VISUALISIEREN
output_dir = os.path.join(workDir, "_Runs", "03_Woody_Fire_combined", "Analysis")
os.makedirs(output_dir, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramm
if len(woody_burned) > 0 and len(woody_not_burned) > 0:
    axes[0].hist(woody_not_burned, bins=50, alpha=0.6, label='Nicht verbrannt', 
                 color='green', density=True)
    axes[0].hist(woody_burned, bins=50, alpha=0.6, label='Verbrannt', 
                 color='red', density=True)
    axes[0].set_xlabel('Woody Cover (%)')
    axes[0].set_ylabel('Dichte')
    axes[0].set_title(f'Woody Cover Verteilung {analysis_year}')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

# Box Plot
data_for_plot = []
labels_for_plot = []
if len(woody_not_burned) > 0:
    data_for_plot.append(woody_not_burned)
    labels_for_plot.append('Nicht verbrannt')
if len(woody_burned) > 0:
    data_for_plot.append(woody_burned)
    labels_for_plot.append('Verbrannt')

if data_for_plot:
    axes[1].boxplot(data_for_plot, labels=labels_for_plot)
    axes[1].set_ylabel('Woody Cover (%)')
    axes[1].set_title(f'Woody Cover Box Plot {analysis_year}')
    axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(output_dir, f"woody_fire_comparison_{analysis_year}.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"\n✓ Visualisierung gespeichert: {plot_path}")

print("\n" + "="*70)
print("✓ ANALYSE ABGESCHLOSSEN!")
print("="*70)

FEUER-WOODY COVER BEZIEHUNGS-ANALYSE

Analysiere Brand-Jahr: 2020

1. DATENÜBERSICHT 2020
   Woody Cover: Band 36
   Burned: Band 281
   Count: Band 282

2. WOODY COVER VERGLEICH
----------------------------------------------------------------------
Verbrannte Pixel: 196,748
Nicht-verbrannte Pixel: 101,069,032

Woody Cover in verbrannten Gebieten:
  Mittelwert: 14.33%
  Median: 6.00%
  Min-Max: 0.00% - 99.00%

Woody Cover in nicht-verbrannten Gebieten:


a:\_BioGeo\butzerfe\miniforge3\envs\wilde\Lib\site-packages\numpy\core\fromnumeric.py:771: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Mittelwert: 16.81%
  Median: 0.00%
  Min-Max: 0.00% - 99.00%

✓ Visualisierung gespeichert: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\03_Woody_Fire_combined\Analysis\woody_fire_comparison_2020.png

✓ ANALYSE ABGESCHLOSSEN!
